<a href="https://colab.research.google.com/github/bhumanivamshi13-cmd/last/blob/main/whether_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install requests pandas

In [8]:
import requests
import pandas as pd

api_key = "e68b0c3bfb3b421350dd0b14bdc2b8aa"
city = "London"

url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

response = requests.get(url)
data = response.json()

print(data)  # see raw JSON

{'coord': {'lon': -0.1257, 'lat': 51.5085}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04n'}], 'base': 'stations', 'main': {'temp': 3.83, 'feels_like': -0.66, 'temp_min': 3.09, 'temp_max': 4.86, 'pressure': 1008, 'humidity': 71, 'sea_level': 1008, 'grnd_level': 1004}, 'visibility': 10000, 'wind': {'speed': 6.17, 'deg': 290, 'gust': 12.86}, 'clouds': {'all': 75}, 'dt': 1771301237, 'sys': {'type': 2, 'id': 2075535, 'country': 'GB', 'sunrise': 1771312265, 'sunset': 1771348695}, 'timezone': 0, 'id': 2643743, 'name': 'London', 'cod': 200}


In [9]:
weather = {
    "city": data["name"],
    "temperature": data["main"]["temp"],
    "humidity": data["main"]["humidity"],
    "pressure": data["main"]["pressure"],
    "wind_speed": data["wind"]["speed"],
    "weather": data["weather"][0]["description"]
}

df = pd.DataFrame([weather])
print(df)

     city  temperature  humidity  pressure  wind_speed        weather
0  London         3.83        71      1008        6.17  broken clouds


In [10]:
df.to_csv("weather_data.csv", index=False)

In [11]:
cities = ["London", "Leicester", "Manchester", "Birmingham", "New York", "Tokyo"]

rows = []

for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    data = requests.get(url).json()

    rows.append({
        "city": data["name"],
        "temp": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "pressure": data["main"]["pressure"],
        "wind_speed": data["wind"]["speed"],
        "weather": data["weather"][0]["main"]
    })

df = pd.DataFrame(rows)
df.to_csv("weather_cities.csv", index=False)

In [12]:
df = pd.read_csv("weather_data.csv")
df.to_parquet("weather_data.parquet", index=False)

In [13]:
import csv

with open("weather_large.csv", "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["city","temp","humidity","pressure","wind_speed","weather"])

    for city in cities:
        data = requests.get(url).json()
        writer.writerow([
            data["name"],
            data["main"]["temp"],
            data["main"]["humidity"],
            data["main"]["pressure"],
            data["wind"]["speed"],
            data["weather"][0]["main"]
        ])

In [14]:
df = pd.read_csv("weather_large.csv")
df.to_parquet("weather_large.parquet", index=False)

In [15]:
import requests
import csv
import time
from datetime import datetime, timedelta

API_KEY = "e68b0c3bfb3b421350dd0b14bdc2b8aa"

# Cities with coordinates (faster & reliable)
cities = {
    "London": (51.5074, -0.1278),
    "Leicester": (52.6369, -1.1398),
    "Manchester": (53.4808, -2.2426),
    "Birmingham": (52.4862, -1.8904),
    "New York": (40.7128, -74.0060),
    "Tokyo": (35.6895, 139.6917)
}

with open("weather_large.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "city","datetime","temp","humidity",
        "pressure","wind_speed","clouds","weather"
    ])

    for city, (lat, lon) in cities.items():
        for days_back in range(1, 30):  # collect past 30 days
            dt = int((datetime.utcnow() - timedelta(days=days_back)).timestamp())

            url = f"https://api.openweathermap.org/data/2.5/onecall/timemachine"
            params = {"lat": lat, "lon": lon, "dt": dt, "appid": API_KEY, "units":"metric"}

            response = requests.get(url, params=params).json()

            if "hourly" in response:
                for hour in response["hourly"]:
                    writer.writerow([
                        city,
                        datetime.utcfromtimestamp(hour["dt"]),
                        hour["temp"],
                        hour["humidity"],
                        hour["pressure"],
                        hour["wind_speed"],
                        hour["clouds"],
                        hour["weather"][0]["main"]
                    ])

            time.sleep(1)  # avoid rate limits

/tmp/ipython-input-2670745863.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  dt = int((datetime.utcnow() - timedelta(days=days_back)).timestamp())


In [18]:
cities = {
    "London": (51.5074, -0.1278),
    "Leicester": (52.6369, -1.1398),
    "Manchester": (53.4808, -2.2426),
    "Birmingham": (52.4862, -1.8904),
    "New York": (40.7128, -74.0060),
    "Tokyo": (35.6895, 139.6917),
    "Delhi": (28.7041, 77.1025),
    "Sydney": (-33.8688, 151.2093),
    "Paris": (48.8566, 2.3522),
    "Dubai": (25.2048, 55.2708)
}

In [19]:
cities = {
    "London": (51.5074, -0.1278),
    "Leicester": (52.6369, -1.1398),
    "Manchester": (53.4808, -2.2426),
    "Birmingham": (52.4862, -1.8904),
    "Liverpool": (53.4084, -2.9916),
    "Leeds": (53.8008, -1.5491),
    "Glasgow": (55.8642, -4.2518),
    "Edinburgh": (55.9533, -3.1883),
    "Cardiff": (51.4816, -3.1791),
    "Belfast": (54.5973, -5.9301),

    "New York": (40.7128, -74.0060),
    "Los Angeles": (34.0522, -118.2437),
    "Chicago": (41.8781, -87.6298),
    "Houston": (29.7604, -95.3698),
    "Phoenix": (33.4484, -112.0740),

    "Toronto": (43.6532, -79.3832),
    "Vancouver": (49.2827, -123.1207),
    "Montreal": (45.5017, -73.5673),

    "Paris": (48.8566, 2.3522),
    "Berlin": (52.5200, 13.4050),
    "Madrid": (40.4168, -3.7038),
    "Rome": (41.9028, 12.4964),
    "Amsterdam": (52.3676, 4.9041),
    "Brussels": (50.8503, 4.3517),
    "Vienna": (48.2082, 16.3738),
    "Zurich": (47.3769, 8.5417),

    "Dubai": (25.2048, 55.2708),
    "Doha": (25.2854, 51.5310),
    "Riyadh": (24.7136, 46.6753),

    "Delhi": (28.7041, 77.1025),
    "Mumbai": (19.0760, 72.8777),
    "Bangalore": (12.9716, 77.5946),
    "Hyderabad": (17.3850, 78.4867),

    "Beijing": (39.9042, 116.4074),
    "Shanghai": (31.2304, 121.4737),
    "Tokyo": (35.6895, 139.6917),
    "Seoul": (37.5665, 126.9780),

    "Sydney": (-33.8688, 151.2093),
    "Melbourne": (-37.8136, 144.9631),
    "Auckland": (-36.8509, 174.7645),

    "Cape Town": (-33.9249, 18.4241),
    "Johannesburg": (-26.2041, 28.0473),
    "Nairobi": (-1.2921, 36.8219),

    "Sao Paulo": (-23.5505, -46.6333),
    "Rio de Janeiro": (-22.9068, -43.1729),
    "Buenos Aires": (-34.6037, -58.3816),
    "Mexico City": (19.4326, -99.1332)
}

In [24]:
import time
time.sleep(1)

In [20]:
import requests
import csv
import time
from datetime import datetime, timedelta, timezone

API_KEY = "e68b0c3bfb3b421350dd0b14bdc2b8aa" # <--- REPLACE WITH YOUR VALID API KEY

# Assuming 'cities' variable is defined in a previous cell (e.g., u07W-a23ofG6)
# cities = { ... }

# Let's encapsulate the whole process into this cell for the full 365 days.
with open("weather_large.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "city","datetime","temp","humidity",
        "pressure","wind_speed","clouds","weather"
    ])

    for city_name, (lat, lon) in cities.items():
        for days_back in range(1, 365):  # collect past 365 days
            # Use timezone-aware objects for current UTC time
            dt = int((datetime.now(timezone.utc) - timedelta(days=days_back)).timestamp())

            url = f"https://api.openweathermap.org/data/2.5/onecall/timemachine"
            params = {"lat": lat, "lon": lon, "dt": dt, "appid": API_KEY, "units":"metric"}

            response = requests.get(url, params=params).json()

            # Check for API key error as seen in kernel state
            if response.get("cod") == 401:
                print(f"API Key Error for {city_name}: {response.get('message', 'Unknown error')}")
                break # Stop processing if API key is invalid

            if "hourly" in response:
                for hour in response["hourly"]:
                    writer.writerow([
                        city_name,
                        datetime.utcfromtimestamp(hour["dt"]),
                        hour["temp"],
                        hour["humidity"],
                        hour["pressure"],
                        hour["wind_speed"],
                        hour["clouds"],
                        hour["weather"][0]["main"]
                    ])

            time.sleep(1)  # Avoid rate limits - crucial for large number of requests

API Key Error for London: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Leicester: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Manchester: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Birmingham: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Liverpool: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Leeds: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Glasgow: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Edinburgh: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
API Key Error for Cardiff: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
A

In [21]:
api_key = "e68b0c3bfb3b421350dd0b14bdc2b8aa"

In [22]:
url = "https://api.openweathermap.org/data/2.5/weather"

In [52]:
api_key = "e68b0c3bfb3b421350dd0b14bdc2b8aa"

In [53]:
url = "https://api.openweathermap.org/data/2.5/weather"

In [23]:
import requests

api_key = "e68b0c3bfb3b421350dd0b14bdc2b8aa"

url = f"https://api.openweathermap.org/data/2.5/weather?q=London,UK&appid={api_key}"

response = requests.get(url)
print(response.status_code)
print(response.text)

200
{"coord":{"lon":-0.1257,"lat":51.5085},"weather":[{"id":800,"main":"Clear","description":"clear sky","icon":"01n"}],"base":"stations","main":{"temp":276.98,"feels_like":272.08,"temp_min":276.24,"temp_max":278.01,"pressure":1009,"humidity":71,"sea_level":1009,"grnd_level":1004},"visibility":10000,"wind":{"speed":7.2,"deg":290,"gust":12.35},"clouds":{"all":8},"dt":1771300920,"sys":{"type":2,"id":2075535,"country":"GB","sunrise":1771312265,"sunset":1771348695},"timezone":0,"id":2643743,"name":"London","cod":200}


In [55]:
!pip install pandas scikit-learn requests

In [24]:
import requests
import pandas as pd
from datetime import datetime

API_KEY = "e68b0c3bfb3b421350dd0b14bdc2b8aa"

In [25]:
cities = [
    "London,UK","Leicester,UK","Manchester,UK","Birmingham,UK","Liverpool,UK",
    "Leeds,UK","Glasgow,UK","Edinburgh,UK","Cardiff,UK","Belfast,UK",
    "New York,US","Los Angeles,US","Chicago,US","Toronto,CA","Delhi,IN",
    "Mumbai,IN","Tokyo,JP","Beijing,CN","Sydney,AU","Paris,FR"
]

In [26]:
weather_data = []

for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&units=metric&appid={API_KEY}"
    response = requests.get(url).json()

    if response.get("cod") != 200:
        print(f"Error for {city}: {response.get('message')}")
        continue

    record = {
        "city": city,
        "temperature": response["main"]["temp"],
        "feels_like": response["main"]["feels_like"],
        "humidity": response["main"]["humidity"],
        "pressure": response["main"]["pressure"],
        "wind_speed": response["wind"]["speed"],
        "weather": response["weather"][0]["main"],
        "timestamp": datetime.utcfromtimestamp(response["dt"])
    }

    weather_data.append(record)

print("Data collected:", len(weather_data))

/tmp/ipython-input-3004873347.py:19: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "timestamp": datetime.utcfromtimestamp(response["dt"])


Data collected: 20


In [27]:
df = pd.DataFrame(weather_data)
df.to_csv("weather_dataset.csv", index=False)
df.head()

,city,temperature,feels_like,humidity,pressure,wind_speed,weather,timestamp
0,"London,UK",3.83,-0.66,71,1008,6.17,Clouds,2026-02-17 04:12:40
1,"Leicester,UK",1.02,-3.73,82,1009,5.17,Clouds,2026-02-17 04:12:54
2,"Manchester,UK",2.01,-2.20,74,1010,4.63,Clouds,2026-02-17 04:09:32
3,"Birmingham,UK",1.53,-2.97,86,1010,4.94,Clouds,2026-02-17 04:12:54
4,"Liverpool,UK",3.12,-1.34,81,1009,5.66,Clear,2026-02-17 04:12:55


In [28]:
df = pd.read_csv("weather_dataset.csv")

df["weather"] = df["weather"].astype("category").cat.codes

X = df[["temperature","feels_like","humidity","pressure","wind_speed"]]
y = df["weather"]

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [31]:
from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression(max_iter=200)
model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, pred1))

Logistic Regression Accuracy: 0.5


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [32]:
from sklearn.tree import DecisionTreeClassifier

model2 = DecisionTreeClassifier()
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, pred2))

Decision Tree Accuracy: 0.75


In [33]:
from sklearn.ensemble import RandomForestClassifier

model3 = RandomForestClassifier()
model3.fit(X_train, y_train)
pred3 = model3.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, pred3))

Random Forest Accuracy: 0.75


In [34]:
from sklearn.svm import SVC

model4 = SVC()
model4.fit(X_train, y_train)
pred4 = model4.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, pred4))

SVM Accuracy: 0.75


In [35]:
results = pd.DataFrame({
    "Model": ["Logistic Regression","Decision Tree","Random Forest","SVM"],
    "Accuracy": [
        accuracy_score(y_test, pred1),
        accuracy_score(y_test, pred2),
        accuracy_score(y_test, pred3),
        accuracy_score(y_test, pred4)
    ]
})

results.to_csv("model_results.csv", index=False)
results

,Model,Accuracy
0,Logistic Regression,0.50
1,Decision Tree,0.75
2,Random Forest,0.75
3,SVM,0.75


In [2]:
import pandas as pd

df = pd.read_csv("weather_large.csv")

big_df = pd.concat([df]*200, ignore_index=True)  # duplicate rows

big_df.to_csv("weather_1GB.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'weather_large.csv'

In [41]:
df = pd.read_csv("weather_1GB.csv")
df.to_parquet("weather_1GB.parquet", index=False)

In [42]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigWeatherData") \
    .config("spark.driver.memory","8g") \
    .getOrCreate()

df = spark.read.parquet("weather_1GB.parquet")
print(df.count())

0


In [43]:
from pyspark.sql.functions import when, col

# create rain label (1 = rain, 0 = no rain)
df = df.withColumn(
    "label",
    when(col("weather").isin(["Rain","Drizzle","Thunderstorm"]), 1).otherwise(0)
)

In [44]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

# Convert weather text to numeric
indexer = StringIndexer(inputCol="weather", outputCol="weather_index")

df = indexer.fit(df).transform(df)

# Feature vector
assembler = VectorAssembler(
    inputCols=["temp","humidity","pressure","wind_speed","clouds","weather_index"],
    outputCol="features"
)

df = assembler.transform(df)

# scaling improves model performance
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
df = scaler.fit(df).transform(df)

IllegalArgumentException: requirement failed: The input column weather must be either string type or numeric type, but got NullType.